# 📊 Tango Benchmark — **Combiner & Analysis Notebook**

This notebook combines outputs from Notebooks A, B, and C and produces:
- Best Adam LR (aggregated across all seeds)
- Per-optimizer summary statistics (mean ± std val_loss, val_acc)
- Win/loss comparison table
- Learning curves plot (val loss vs epoch for each optimizer)
- Final JSON report saved to `tango_benchmark_final.json`

### Instructions:
1. Upload `results_A.txt`, `results_B.txt`, `results_C.txt` to this Colab session.
   - Use the **Files panel** (folder icon in left sidebar) → Upload button
2. Run all cells.

> **No GPU needed** — this notebook only analyzes and plots.

In [ ]:
!pip install -q matplotlib numpy
import json, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict

print('Combiner ready ✓')

In [ ]:
# ── Cell 2: Load result files ──────────────────────────────────────────────────
RESULT_FILES = ['results_A.txt', 'results_B.txt', 'results_C.txt']

notebooks = []
for fname in RESULT_FILES:
    if not os.path.exists(fname):
        print(f'⚠️  {fname} NOT FOUND — upload it to Colab Files first!')
        continue
    with open(fname) as f:
        data = json.load(f)
    notebooks.append(data)
    print(f'✓ Loaded {fname}  (notebook={data["notebook"]}, seed={data["seed"]})')

if len(notebooks) < 3:
    print(f'\n⚠️  Only {len(notebooks)}/3 notebooks loaded. Results will be partial.')
else:
    print(f'\n✅ All 3 notebooks loaded!')

In [ ]:
# ── Cell 3: Find best Adam LR (across all seeds & notebooks) ──────────────────
#
# For each LR, average the best_val_loss across all seeds that ran it.
# The LR with the lowest average best_val_loss is declared best.

lr_losses = defaultdict(list)  # lr_str -> [best_val_loss, ...]

for nb in notebooks:
    for r in nb['adam']:
        lr_key = float(r['lr'])
        lr_losses[lr_key].append(r['best_val_loss'])

print('Adam LR grid results (mean best_val_loss across seeds):')
print(f'{"LR":>10}  {"N seeds":>8}  {"Mean best_val_loss":>20}  {"Std":>8}')
print('-' * 55)

best_lr     = None
best_lr_loss = float('inf')

lr_summary = {}
for lr_val, losses in sorted(lr_losses.items(), reverse=True):
    mean_l = np.mean(losses)
    std_l  = np.std(losses)
    lr_summary[lr_val] = {'mean': mean_l, 'std': std_l, 'n': len(losses), 'losses': losses}
    marker = '  ← best' if mean_l < best_lr_loss else ''
    print(f'{lr_val:>10.0e}  {len(losses):>8}  {mean_l:>20.4f}  {std_l:>8.4f}{marker}')
    if mean_l < best_lr_loss:
        best_lr_loss = mean_l
        best_lr      = lr_val

print(f'\n🏆 Best Adam LR: {best_lr:.0e}  (mean best_val_loss = {best_lr_loss:.4f})')

In [ ]:
# ── Cell 4: Collect per-optimizer summary stats ───────────────────────────────
# Adam → only the runs at best_lr
# Lion → all runs (each notebook ran 1 seed)
# Tango → all runs

adam_best_runs  = []
lion_runs       = []
tango_runs      = []

for nb in notebooks:
    for r in nb['adam']:
        if abs(float(r['lr']) - best_lr) < 1e-12:
            adam_best_runs.append(r)
    for r in nb['lion']:
        lion_runs.append(r)
    for r in nb['tango']:
        tango_runs.append(r)

def summarise(runs, label):
    if not runs:
        print(f'  {label}: NO DATA')
        return {}
    final_accs   = [r['final_val_acc']  for r in runs]
    best_accs    = [r['best_val_acc']   for r in runs]
    final_losses = [r['final_val_loss'] for r in runs]
    best_losses  = [r['best_val_loss']  for r in runs]
    print(f'  {label} ({len(runs)} runs)')
    print(f'    best_val_loss : mean={np.mean(best_losses):.4f}  std={np.std(best_losses):.4f}  '
          f'min={np.min(best_losses):.4f}  max={np.max(best_losses):.4f}')
    print(f'    best_val_acc  : mean={np.mean(best_accs):.4f}  std={np.std(best_accs):.4f}  '
          f'min={np.min(best_accs):.4f}  max={np.max(best_accs):.4f}')
    return {'label': label, 'n': len(runs),
            'best_val_loss': {'mean': np.mean(best_losses), 'std': np.std(best_losses),
                              'min': np.min(best_losses),   'max': np.max(best_losses), 'values': best_losses},
            'best_val_acc':  {'mean': np.mean(best_accs),  'std': np.std(best_accs),
                              'min': np.min(best_accs),     'max': np.max(best_accs),  'values': best_accs},
            'final_val_loss': {'mean': np.mean(final_losses), 'std': np.std(final_losses)},
            'final_val_acc':  {'mean': np.mean(final_accs),   'std': np.std(final_accs)},
           }

print('\n' + '='*60)
print('  PER-OPTIMIZER SUMMARY')
print('='*60)
adam_stats  = summarise(adam_best_runs,  f'Adam (lr={best_lr:.0e})')
print()
lion_stats  = summarise(lion_runs,  'Lion')
print()
tango_stats = summarise(tango_runs, 'Tango')

In [ ]:
# ── Cell 5: Win/loss table (per-seed) ────────────────────────────────────────
print('\n' + '='*60)
print('  SEED-BY-SEED COMPARISON (best_val_acc)')
print('='*60)
print(f'{"Seed":>6}  {"Adam":>8}  {"Lion":>8}  {"Tango":>8}  {"Winner":>8}')
print('-' * 50)

# Build seed-indexed dicts
def by_seed(runs):
    return {r['seed']: r for r in runs}

adam_by_seed  = by_seed(adam_best_runs)
lion_by_seed  = by_seed(lion_runs)
tango_by_seed = by_seed(tango_runs)

all_seeds = sorted(set(list(adam_by_seed) + list(lion_by_seed) + list(tango_by_seed)))
tango_wins = lion_wins = adam_wins = 0

for seed in all_seeds:
    a = adam_by_seed.get(seed,  {}).get('best_val_acc', float('nan'))
    l = lion_by_seed.get(seed,  {}).get('best_val_acc', float('nan'))
    t = tango_by_seed.get(seed, {}).get('best_val_acc', float('nan'))
    vals = {'Adam': a, 'Lion': l, 'Tango': t}
    valid = {k: v for k, v in vals.items() if not np.isnan(v)}
    winner = max(valid, key=valid.get) if valid else 'N/A'
    if winner == 'Tango': tango_wins += 1
    elif winner == 'Lion': lion_wins += 1
    elif winner == 'Adam': adam_wins += 1
    print(f'{seed:>6}  {a:>8.4f}  {l:>8.4f}  {t:>8.4f}  {winner:>8}')

n = len(all_seeds)
print('-' * 50)
print(f'Wins   {adam_wins:>8}  {lion_wins:>8}  {tango_wins:>8}  (out of {n})')

In [ ]:
# ── Cell 6: Tango-specific stats ─────────────────────────────────────────────
if tango_runs:
    tang_rates = [r.get('tang_rate', 0) for r in tango_runs]
    print('\nTango tangent fire stats:')
    for r in tango_runs:
        seed = r['seed']; te = r.get('tang_exec',0); tb = r.get('tang_block',0)
        tr = r.get('tang_rate', te/max(te+tb,1))
        print(f'  seed={seed}  exec={te}  blocked={tb}  fire_rate={tr:.2%}')
    print(f'  Mean fire rate: {np.mean(tang_rates):.2%}')

In [ ]:
# ── Cell 7: Learning curves plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Tango Benchmark — ResNet-18 on CIFAR-10 (50 epochs)', fontsize=13, fontweight='bold')

COLORS = {'Adam': '#2196F3', 'Lion': '#FF9800', 'Tango': '#4CAF50'}
ALPHA_SEED = 0.35

def plot_curves(ax, runs_dict, metric, ylabel, title):
    for opt_label, runs in runs_dict.items():
        color = COLORS.get(opt_label, 'gray')
        histories = []
        for r in runs:
            hist = r.get('history_summary', [])
            if hist:
                epochs = [h['epoch'] for h in hist]
                vals   = [h[metric]  for h in hist]
                ax.plot(epochs, vals, color=color, alpha=ALPHA_SEED, linewidth=0.8)
                histories.append(vals)
        # Mean line
        if histories:
            min_len = min(len(h) for h in histories)
            arr = np.array([h[:min_len] for h in histories])
            mean_vals = arr.mean(axis=0)
            ep = list(range(1, min_len + 1))
            ax.plot(ep, mean_vals, color=color, linewidth=2.2, label=f'{opt_label} (mean)')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)

all_runs = {
    'Adam':  adam_best_runs,
    'Lion':  lion_runs,
    'Tango': tango_runs,
}

plot_curves(axes[0], all_runs, 'val_loss', 'Validation Loss', 'Val Loss vs Epoch')
plot_curves(axes[1], all_runs, 'val_acc',  'Validation Accuracy', 'Val Acc vs Epoch')

plt.tight_layout()
plt.savefig('tango_benchmark_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved: tango_benchmark_curves.png')

In [ ]:
# ── Cell 8: Bar chart comparison ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Optimizer Comparison Summary (mean ± std across seeds)', fontsize=12, fontweight='bold')

opts   = ['Adam', 'Lion', 'Tango']
colors = [COLORS[o] for o in opts]
stats  = [adam_stats, lion_stats, tango_stats]

for ax, metric, ylabel, higher_is_better in [
    (axes[0], 'best_val_loss', 'Best Val Loss (↓)', False),
    (axes[1], 'best_val_acc',  'Best Val Acc (↑)',  True),
]:
    means = [s[metric]['mean'] if s else 0 for s in stats]
    stds  = [s[metric]['std']  if s else 0 for s in stats]
    bars  = ax.bar(opts, means, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax.errorbar(opts, means, yerr=stds, fmt='none', color='black', capsize=5, linewidth=1.5)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (max(means)*0.01),
                f'{mean:.4f}', ha='center', va='bottom', fontsize=9)
    # Highlight best
    best_idx = np.argmin(means) if not higher_is_better else np.argmax(means)
    bars[best_idx].set_edgecolor('gold'); bars[best_idx].set_linewidth(2.5)
    ax.set_ylabel(ylabel); ax.set_title(ylabel); ax.grid(axis='y', alpha=0.3)
    # Tight y-range
    pad = max(stds) * 3 if max(stds) > 0 else 0.01
    ax.set_ylim(max(0, min(means) - pad), max(means) + pad * 2)

plt.tight_layout()
plt.savefig('tango_benchmark_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('Bar chart saved: tango_benchmark_bars.png')

In [ ]:
# ── Cell 9: Print final summary ───────────────────────────────────────────────
print('\n' + '='*65)
print('  TANGO BENCHMARK FINAL SUMMARY')
print('  ResNet-18 | CIFAR-10 | 50 epochs | batch=128 | 3 seeds')
print('='*65)

rows = []
for label, s in [('Adam (best LR)', adam_stats), ('Lion', lion_stats), ('Tango', tango_stats)]:
    if not s: continue
    rows.append({
        'label': label,
        'best_acc_mean':  s['best_val_acc']['mean'],
        'best_acc_std':   s['best_val_acc']['std'],
        'best_loss_mean': s['best_val_loss']['mean'],
        'best_loss_std':  s['best_val_loss']['std'],
    })

# Sort by best_acc desc
rows.sort(key=lambda r: -r['best_acc_mean'])

print(f'  {"Rank":<4} {"Optimizer":<18} {"Best Val Acc":>14} {"Best Val Loss":>15}')
print('  ' + '-'*55)
for i, row in enumerate(rows, 1):
    medal = ['🥇','🥈','🥉'][i-1] if i <= 3 else f' {i}.'
    print(f'  {medal}  {row["label"]:<16}  '
          f'{row["best_acc_mean"]:.4f} ± {row["best_acc_std"]:.4f}   '
          f'{row["best_loss_mean"]:.4f} ± {row["best_loss_std"]:.4f}')

print()
if tango_runs:
    tang_rates = [r.get('tang_rate',0) for r in tango_runs]
    print(f'  Tango tangent fire rate: {np.mean(tang_rates):.2%} (mean across seeds)')

# Tango vs Adam margin
if adam_stats and tango_stats:
    delta_acc  = tango_stats['best_val_acc']['mean']  - adam_stats['best_val_acc']['mean']
    delta_loss = tango_stats['best_val_loss']['mean'] - adam_stats['best_val_loss']['mean']
    print(f'\n  Tango vs Adam  Δacc={delta_acc:+.4f}  Δloss={delta_loss:+.4f}')
if lion_stats and tango_stats:
    delta_acc  = tango_stats['best_val_acc']['mean']  - lion_stats['best_val_acc']['mean']
    delta_loss = tango_stats['best_val_loss']['mean'] - lion_stats['best_val_loss']['mean']
    print(f'  Tango vs Lion  Δacc={delta_acc:+.4f}  Δloss={delta_loss:+.4f}')

print('='*65)

In [ ]:
# ── Cell 10: Save final JSON report ──────────────────────────────────────────
def make_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)): return float(obj)
    if isinstance(obj, (np.int32, np.int64)):     return int(obj)
    if isinstance(obj, np.ndarray):               return obj.tolist()
    if isinstance(obj, dict):  return {k: make_serializable(v) for k,v in obj.items()}
    if isinstance(obj, list):  return [make_serializable(i) for i in obj]
    return obj

final_report = make_serializable({
    'benchmark': 'tango_resnet18_cifar10',
    'config': {'model': 'ResNet-18', 'dataset': 'CIFAR-10',
               'epochs': 50, 'batch_size': 128, 'seeds': all_seeds},
    'best_adam_lr': best_lr,
    'lr_grid_summary': {str(k): v for k, v in lr_summary.items()},
    'summary': {
        'adam':  adam_stats,
        'lion':  lion_stats,
        'tango': tango_stats,
    },
    'per_seed': {
        'adam':  [{k: v for k,v in r.items() if k != 'history_summary'} for r in adam_best_runs],
        'lion':  [{k: v for k,v in r.items() if k != 'history_summary'} for r in lion_runs],
        'tango': [{k: v for k,v in r.items() if k != 'history_summary'} for r in tango_runs],
    },
    'wins': {'adam': adam_wins, 'lion': lion_wins, 'tango': tango_wins, 'total_seeds': n},
})

with open('tango_benchmark_final.json', 'w') as f:
    json.dump(final_report, f, indent=2)

print('✅ Final report saved: tango_benchmark_final.json')
print('   Download from Colab Files panel (left sidebar).')
print()
print('Outputs generated:')
for fname in ['tango_benchmark_final.json', 'tango_benchmark_curves.png', 'tango_benchmark_bars.png']:
    exists = '✓' if os.path.exists(fname) else '✗ MISSING'
    print(f'  {exists}  {fname}')